# Hospital Readmission Risk - Exploratory Data Analysis (Compact)

**Objective**: Goal-driven EDA to inform preprocessing decisions and model development

**Structure**:
- Section I: Initialization & Binary Target Creation
- Section II: Feature Strategy (selection, engineering, preprocessing)
- Section III: Risk Factor Analysis (clinical insights & fairness)
- Section IV: Implementation Specifications

**Expected Runtime**: ~3-5 minutes

# I. Initialization

## 1. Import Libraries & Set Constants

In [23]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, pointbiserialr

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')

# Constants
DATA_PATH = "../data/diabetic_data.csv"
MAP_PATH = "../data/IDS_mapping.csv"

print("Libraries imported successfully")

Libraries imported successfully


## 2. Load Data

In [24]:
# Load datasets
data = pd.read_csv(DATA_PATH)
mapping = pd.read_csv(MAP_PATH)

print(f"Data loaded successfully")
print(f"Dataset shape: {data.shape}")
print(f"Mapping data shape: {mapping.shape}")

data.head()

Data loaded successfully
Dataset shape: (101766, 50)
Mapping data shape: (67, 2)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


## 3. Create Binary Target Variable

Convert `readmitted` to binary format:
- **1** = Readmitted within 30 days (`<30`)
- **0** = No readmission or readmitted after 30 days (`NO`, `>30`)

In [25]:
# Check original target distribution
print("Original target distribution:")
print(data['readmitted'].value_counts())
print("\nPercentages:")
print(data['readmitted'].value_counts(normalize=True).round(4) * 100)

# Create binary target
data['readmitted_30day'] = (data['readmitted'] == '<30').astype(int)

# Drop original target
data = data.drop('readmitted', axis=1)

# Show new target distribution
print("\n" + "="*50)
print("NEW BINARY TARGET DISTRIBUTION")
print("="*50)
print(f"\nReadmitted within 30 days (1): {data['readmitted_30day'].sum():,} ({data['readmitted_30day'].mean()*100:.2f}%)")
print(f"Not readmitted or >30 days (0): {(data['readmitted_30day']==0).sum():,} ({(1-data['readmitted_30day'].mean())*100:.2f}%)")
print(f"\nClass imbalance ratio: {(1-data['readmitted_30day'].mean()) / data['readmitted_30day'].mean():.2f}:1")

Original target distribution:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

Percentages:
readmitted
NO     53.91
>30    34.93
<30    11.16
Name: proportion, dtype: float64

NEW BINARY TARGET DISTRIBUTION

Readmitted within 30 days (1): 11,357 (11.16%)
Not readmitted or >30 days (0): 90,409 (88.84%)

Class imbalance ratio: 7.96:1


# II. Feature Strategy

This section analyzes all features to determine:
1. **KEEP/DROP Decision**: Which raw features to retain or remove
2. **Feature Engineering**: How to create new features from raw data
3. **Preprocessing Specifications**: How to process kept features

## 1. Dataset Overview

In [26]:
print("="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"\nTotal patients: {len(data):,}")
print(f"Total features (excluding target): {len(data.columns) - 1}")
print(f"Target variable: readmitted_30day")
print(f"  - Positive class (30-day readmission): {data['readmitted_30day'].sum():,} ({data['readmitted_30day'].mean()*100:.2f}%)")
print(f"  - Negative class: {(data['readmitted_30day']==0).sum():,} ({(1-data['readmitted_30day'].mean())*100:.2f}%)")

print(f"\nFeature types:")
print(f"  - Numerical: {len(data.select_dtypes(include=[np.number]).columns) - 1}")  # -1 for target
print(f"  - Categorical: {len(data.select_dtypes(include=['object']).columns)}")

print(f"\nMemory usage: {data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

DATASET OVERVIEW

Total patients: 101,766
Total features (excluding target): 49
Target variable: readmitted_30day
  - Positive class (30-day readmission): 11,357 (11.16%)
  - Negative class: 90,409 (88.84%)

Feature types:
  - Numerical: 13
  - Categorical: 36

Memory usage: 215.22 MB


## 2. Feature Keep/Drop Decision

Determine which raw features to keep or drop based on data quality and predictive value

**Mechanism Explanation:**

For each feature, we calculate:
1. **Type**: Numerical or Categorical based on dtype
2. **Missing%**: Includes NULL values + '?' markers for categorical features
3. **Unique Count/Percentage**: Measures cardinality (important for encoding strategy)
4. **Predictive Strength**: 
   - **Numerical features**: Point-biserial correlation with binary target (range: 0-1)
   - **Categorical features**: Cramér's V statistic (chi-square based, range: 0-1)
   - Higher values indicate stronger association with readmission outcome

In [27]:
# Build feature analysis table
feature_analysis = []

for col in data.columns:
    if col == 'readmitted_30day':
        continue
    
    # Basic info
    dtype = data[col].dtype
    feature_type = 'Numerical' if np.issubdtype(dtype, np.number) else 'Categorical'
    
    # Missing values (including '?' for categorical)
    null_count = data[col].isnull().sum()
    if feature_type == 'Categorical':
        question_mark_count = (data[col] == '?').sum()
        total_missing = null_count + question_mark_count
    else:
        total_missing = null_count
    missing_pct = (total_missing / len(data)) * 100
    
    # Cardinality/Unique%
    unique_count = data[col].nunique()
    unique_pct = (unique_count / len(data)) * 100
    
    # Predictive strength
    try:
        if feature_type == 'Numerical':
            # Point-biserial correlation for numerical features
            valid_mask = ~data[col].isnull()
            if valid_mask.sum() > 0:
                corr, p_value = pointbiserialr(data.loc[valid_mask, 'readmitted_30day'], 
                                               data.loc[valid_mask, col])
                strength = abs(corr)
            else:
                strength = 0.0
        else:
            # Cramér's V for categorical features
            if unique_count > 1 and total_missing < len(data):
                # Remove missing values for chi-square test
                valid_data = data[data[col] != '?'] if '?' in data[col].values else data
                contingency = pd.crosstab(valid_data[col], valid_data['readmitted_30day'])
                chi2, p_value, dof, expected = chi2_contingency(contingency)
                n = contingency.sum().sum()
                strength = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))
            else:
                strength = 0.0
    except:
        strength = 0.0
    
    feature_analysis.append({
        'Feature': col,
        'Type': feature_type,
        'Missing%': round(missing_pct, 2),
        'Unique_Count': unique_count,
        'Unique%': round(unique_pct, 2),
        'Predictive_Strength': round(strength, 4)
    })

# Create DataFrame
feature_df = pd.DataFrame(feature_analysis)

# Sort by predictive strength
feature_df = feature_df.sort_values('Predictive_Strength', ascending=False).reset_index(drop=True)

print(f"Feature analysis complete. Analyzed {len(feature_df)} features.")
feature_df.head(10)

Feature analysis complete. Analyzed 49 features.


,Feature,Type,Missing%,Unique_Count,Unique%,Predictive_Strength
0,number_inpatient,Numerical,0.00,21,0.02,0.1651
1,diag_1,Categorical,0.02,717,0.70,0.1322
2,diag_3,Categorical,1.40,790,0.78,0.1230
3,diag_2,Categorical,0.35,749,0.74,0.1163
4,medical_specialty,Categorical,49.08,73,0.07,0.0809
5,number_emergency,Numerical,0.00,33,0.03,0.0607
6,discharge_disposition_id,Numerical,0.00,26,0.03,0.0506
7,number_diagnoses,Numerical,0.00,16,0.02,0.0495
8,time_in_hospital,Numerical,0.00,14,0.01,0.0442
9,insulin,Categorical,0.00,4,0.00,0.0433


**Decision Logic Explanation:**

The algorithm determines whether to KEEP or DROP each raw feature:

**DROP features if:**
- ID columns (encounter_id, patient_nbr) - not predictive, just identifiers
- Zero/near-zero variance - no information content (e.g., all same value)
- Missing rate >95% - insufficient data for reliable modeling
- Predictive strength <0.01 - negligible association with target

**KEEP features if:**
- Protected attributes (race, gender, age) - required for fairness analysis regardless of predictive power
- Clinical indicators with <95% missing - even high missingness can be informative (A1Cresult, max_glu_serum)
- Sufficient predictive power (≥0.01) and acceptable data quality

Note: Individual medications and diagnosis codes are KEPT as raw features but will be used for feature engineering in the next section.

In [28]:
# Add KEEP/DROP decision logic
def determine_decision(row):
    feature = row['Feature']
    missing_pct = row['Missing%']
    unique_pct = row['Unique%']
    strength = row['Predictive_Strength']
    feature_type = row['Type']
    unique_count = row['Unique_Count']
    
    # ID columns - DROP
    if feature in ['encounter_id', 'patient_nbr']:
        return 'DROP', 'ID column'
    
    # Protected attributes for fairness - KEEP (regardless of predictive strength)
    if feature in ['race', 'gender', 'age']:
        return 'KEEP', 'Protected attribute for fairness analysis'
    
    # Near-zero variance - DROP
    if unique_count == 1 or (unique_count == 2 and missing_pct > 90):
        return 'DROP', 'Zero/near-zero variance'
    
    # Very high missing rate - DROP (>95% threshold, but keep clinically important ones)
    if missing_pct > 95:
        return 'DROP', f'Extremely high missing rate ({missing_pct:.1f}%)'
    
    # Clinical indicators - keep despite high missing (missingness may be informative)
    if feature in ['A1Cresult', 'max_glu_serum']:
        if missing_pct < 95:  # Keep if <95% missing
            return 'KEEP', 'Clinical indicator (high missing but informative)'
    
    # Weak predictors - DROP
    if strength < 0.01:
        return 'DROP', f'Very weak predictor ({strength:.4f})'
    
    # Keep all others (including medications and diagnoses for feature engineering)
    return 'KEEP', ''

# Apply decision logic
feature_df[['Decision', 'Reason']] = feature_df.apply(
    lambda row: pd.Series(determine_decision(row)), axis=1
)

# Add preprocessing specifications for KEEP features
def add_preprocessing_specs(row):
    if row['Decision'] != 'KEEP':
        return ''
    
    feature = row['Feature']
    feature_type = row['Type']
    missing_pct = row['Missing%']
    unique_count = row['Unique_Count']
    
    specs = []
    
    # Special handling for protected attributes
    if feature in ['race', 'gender', 'age']:
        specs.append('Protected: preserve original values')
    
    # Imputation with advanced strategies
    if missing_pct > 0:
        if feature_type == 'Numerical':
            specs.append('Impute: median')
        else:
            # Group-wise imputation for MAR features (Missing At Random)
            if feature == 'medical_specialty':
                specs.append('Impute: mode by admission_type_id')
            elif feature == 'payer_code':
                specs.append('Impute: mode by admission_source_id')
            else:
                specs.append('Impute: mode')
        
        # Missing indicator for high missing%
        if missing_pct > 10:
            specs.append('Create missing_indicator')
    
    # Encoding with advanced strategies
    if feature_type == 'Categorical':
        # Check if ordered categorical (age is ordinal)
        if feature == 'age':
            specs.append('Encode: ordinal (preserve order)')
        elif unique_count < 10:
            specs.append('Encode: one-hot')
        else:
            specs.append('Encode: target (CV-safe + Bayesian smoothing)')
    
    # Outlier handling with validation
    if feature_type == 'Numerical':
        specs.append('Outliers: IQR winsorize + range validation')
    
    return ' | '.join(specs) if specs else 'None'

feature_df['Processing_Specs'] = feature_df.apply(add_preprocessing_specs, axis=1)

print("\n" + "="*60)
print("KEEP/DROP DECISION SUMMARY")
print("="*60)
print(f"\nTotal features analyzed: {len(feature_df)}")
print(f"Features to DROP: {(feature_df['Decision']=='DROP').sum()}")
print(f"Features to KEEP: {(feature_df['Decision']=='KEEP').sum()}")

# Highlight protected attributes
protected = feature_df[feature_df['Feature'].isin(['race', 'gender', 'age'])]
if len(protected) > 0:
    print(f"\n⚠️ Protected attributes for fairness (KEPT): {', '.join(protected['Feature'].tolist())}")

# Identify features that will be used for engineering
medication_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 
                   'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 
                   'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 
                   'miglitol', 'troglitazone', 'tolazamide', 'insulin',
                   'glyburide-metformin', 'glipizide-metformin', 
                   'glimepiride-pioglitazone', 'metformin-rosiglitazone', 
                   'metformin-pioglitazone', 'examide', 'citoglipton']
diag_cols = ['diag_1', 'diag_2', 'diag_3']

kept_meds = [m for m in medication_cols if m in feature_df[feature_df['Decision']=='KEEP']['Feature'].values]
kept_diags = [d for d in diag_cols if d in feature_df[feature_df['Decision']=='KEEP']['Feature'].values]

if kept_meds or kept_diags:
    print(f"\n📋 Features available for engineering:")
    if kept_meds:
        print(f"   - Medication columns: {len(kept_meds)} features")
    if kept_diags:
        print(f"   - Diagnosis columns: {len(kept_diags)} features")


KEEP/DROP DECISION SUMMARY

Total features analyzed: 49
Features to DROP: 23
Features to KEEP: 26

⚠️ Protected attributes for fairness (KEPT): age, race, gender

📋 Features available for engineering:
   - Medication columns: 4 features
   - Diagnosis columns: 3 features


**Preprocessing Specifications Logic:**

For **KEEP** features, specifications are automatically determined:

**Imputation Strategy:**
- Numerical features with missing data → Median imputation (robust to outliers)
- Categorical features with missing data → Mode imputation (most frequent value)
- **Group-wise imputation for MAR** (Missing At Random):
  - `medical_specialty` → Mode by `admission_type_id` (context-aware)
  - `payer_code` → Mode by `admission_source_id` (context-aware)
- Features with >10% missing → Create binary missing indicator (missingness may be informative)

**Encoding Strategy:**
- **Ordinal features** (age) → Preserve natural ordering (not one-hot)
- Categorical with <10 unique values → One-hot encoding (manageable dimensionality)
- Categorical with ≥10 unique values → **Target encoding with Bayesian smoothing** (prevents overfitting on rare categories + CV-safe)

**Outlier Handling:**
- All numerical features → **IQR-based winsorization + range validation** (cap extreme values at 1.5×IQR bounds + enforce domain constraints)

These specifications will guide the preprocessing pipeline implementation.

In [29]:
# Display complete feature decision table
print("="*100)
print("COMPREHENSIVE KEEP/DROP DECISION TABLE")
print("="*100)

# Display by decision type
for decision in ['DROP', 'KEEP']:
    subset = feature_df[feature_df['Decision'] == decision]
    if len(subset) > 0:
        print(f"\n{'='*100}")
        print(f"{decision} FEATURES ({len(subset)} features)")
        print('='*100)
        if decision == 'KEEP':
            print(subset[['Feature', 'Type', 'Missing%', 'Unique_Count', 'Predictive_Strength', 
                         'Processing_Specs']].to_string(index=False))
        else:
            print(subset[['Feature', 'Type', 'Missing%', 'Unique_Count', 'Predictive_Strength', 
                         'Reason']].to_string(index=False))

COMPREHENSIVE KEEP/DROP DECISION TABLE

DROP FEATURES (23 features)
                 Feature        Type  Missing%  Unique_Count  Predictive_Strength                              Reason
                  weight Categorical     96.86            10               0.0383 Extremely high missing rate (96.9%)
            encounter_id   Numerical      0.00        101766               0.0085                           ID column
             glimepiride Categorical      0.00             4               0.0083        Very weak predictor (0.0083)
             patient_nbr   Numerical      0.00         71518               0.0079                           ID column
            pioglitazone Categorical      0.00             4               0.0079        Very weak predictor (0.0079)
           rosiglitazone Categorical      0.00             4               0.0073        Very weak predictor (0.0073)
                miglitol Categorical      0.00             4               0.0071        Very weak predict

## 3. Feature Engineering Strategy

Transform raw KEPT features into composite features that better capture clinical patterns. These specifications are based on proven strategies from the preprocessing pipeline.

**Engineering Categories:**
1. **Diagnosis Aggregation**: Group high-cardinality diagnosis codes into clinical categories
2. **Medication Composites**: Aggregate 23 individual medication flags into meaningful metrics
3. **Utilization Patterns**: Combine encounter counts into utilization profiles
4. **Clinical Complexity**: Create composite indicators of patient complexity

In [30]:
# Define engineered features from raw KEPT features
engineered_features = []

# ========================================
# 1. DIAGNOSIS AGGREGATION
# ========================================
# Problem: diag_1, diag_2, diag_3 have 700+ unique values each (unstable for target encoding)
# Solution: Aggregate into 9 clinical categories based on ICD-9 code ranges

print("="*100)
print("FEATURE ENGINEERING SPECIFICATIONS")
print("="*100)

print("\n" + "="*100)
print("1. DIAGNOSIS AGGREGATION (9 clinical categories)")
print("="*100)
print("\nSource features: diag_1, diag_2, diag_3 (717-790 unique values each)")
print("\nEngineered features:")

diag_categories = {
    'diag_1_circulatory': 'ICD-9 390-459, 785 (heart disease, hypertension, stroke)',
    'diag_1_respiratory': 'ICD-9 460-519, 786 (COPD, pneumonia, asthma)',
    'diag_1_diabetes': 'ICD-9 250.xx (diabetes complications)',
    'diag_1_digestive': 'ICD-9 520-579, 787 (GI disorders)',
    'diag_1_injury': 'ICD-9 800-999 (fractures, trauma)',
    'diag_1_musculoskeletal': 'ICD-9 710-739 (arthritis, back pain)',
    'diag_1_genitourinary': 'ICD-9 580-629, 788 (kidney, urinary)',
    'diag_1_neoplasms': 'ICD-9 140-239 (cancer)',
    'diag_1_other': 'All other ICD-9 codes'
}

for feat, desc in diag_categories.items():
    print(f"  • {feat:30s} = {desc}")
    engineered_features.append({
        'Feature': feat,
        'Type': 'Binary',
        'Source': 'diag_1',
        'Description': desc,
        'Preprocessing': 'Binary flag | No further encoding needed'
    })

print(f"\n  [Repeat same categories for diag_2 and diag_3]")
print(f"  Total diagnosis features: 9 × 3 = 27 binary flags")

# Add diag_2 and diag_3 features (same structure)
for diag_num in [2, 3]:
    for category in ['circulatory', 'respiratory', 'diabetes', 'digestive', 'injury', 
                     'musculoskeletal', 'genitourinary', 'neoplasms', 'other']:
        engineered_features.append({
            'Feature': f'diag_{diag_num}_{category}',
            'Type': 'Binary',
            'Source': f'diag_{diag_num}',
            'Description': f'Same as diag_1_{category}',
            'Preprocessing': 'Binary flag | No further encoding needed'
        })

# ========================================
# 2. MEDICATION FEATURES (KEPT AS-IS)
# ========================================
print("\n" + "="*100)
print("2. MEDICATION FEATURES (keep individual columns)")
print("="*100)
print("\nSource features: metformin, repaglinide, glipizide, insulin, etc.")
print("\nStrategy: KEEP all individual medication features without aggregation")
print("  • Preserves specific medication information (each drug has unique properties)")
print("  • Model can learn medication-specific effects on readmission risk")
print("  • Preprocessing: Already specified in Section II.2 (one-hot encoding for 4-level categorical)")

medication_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 
                   'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 
                   'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 
                   'miglitol', 'troglitazone', 'tolazamide', 'insulin',
                   'glyburide-metformin', 'glipizide-metformin', 
                   'glimepiride-pioglitazone', 'metformin-rosiglitazone', 
                   'metformin-pioglitazone', 'examide', 'citoglipton']

kept_meds = [m for m in medication_cols if m in feature_df[feature_df['Decision']=='KEEP']['Feature'].values]
print(f"\n  Total medication features kept: {len(kept_meds)}")

# ========================================
# 3. UTILIZATION FEATURES (KEPT AS-IS)
# ========================================
print("\n" + "="*100)
print("3. UTILIZATION FEATURES (keep original counts)")
print("="*100)
print("\nSource features: number_inpatient, number_emergency, number_outpatient")
print("\nStrategy: KEEP as raw counts without aggregation")
print("  • Each encounter type has distinct clinical meaning:")
print("    - number_inpatient: Hospital admissions (most severe)")
print("    - number_emergency: ER visits (acute care)")
print("    - number_outpatient: Clinic visits (routine care)")
print("  • Model can learn type-specific utilization patterns")
print("  • Preprocessing: Already specified in Section II.2 (IQR winsorize + range validation)")

util_features = ['number_inpatient', 'number_emergency', 'number_outpatient']
print(f"\n  Total utilization features kept: {len(util_features)}")

# ========================================
# 4. AGE STRATIFICATION (multiple buckets)
# ========================================
print("\n" + "="*100)
print("4. AGE STRATIFICATION (granular age groups)")
print("="*100)
print("\nSource feature: age (10 brackets: [0-10), [10-20), ..., [90-100))")
print("\nEngineered features:")

age_features = [
    ('age_bucket', 'Ordinal encoding: [0-10)→0, [10-20)→1, ..., [90-100)→9', 'Ordinal'),
    ('age_young', 'Binary: age < 30 (pediatric/young adult)', 'Binary'),
    ('age_adult', 'Binary: 30 ≤ age < 50 (working age)', 'Binary'),
    ('age_middle', 'Binary: 50 ≤ age < 65 (pre-Medicare)', 'Binary'),
    ('age_senior', 'Binary: 65 ≤ age < 80 (Medicare, active senior)', 'Binary'),
    ('age_elderly', 'Binary: age ≥ 80 (frail elderly)', 'Binary')
]

for feat, desc, feat_type in age_features:
    print(f"  • {feat:20s} = {desc}")
    engineered_features.append({
        'Feature': feat,
        'Type': feat_type,
        'Source': 'age',
        'Description': desc,
        'Preprocessing': 'Ordinal encoding (0-9)' if feat_type == 'Ordinal' else 'Binary flag | No encoding needed'
    })

print(f"\n  → Creates 6 age features capturing non-linear age effects")
print(f"  → Allows model to learn age-specific readmission patterns")

# ========================================
# SUMMARY
# ========================================
print("\n" + "="*100)
print("FEATURE COUNT SUMMARY")
print("="*100)

keep_count = (feature_df['Decision']=='KEEP').sum()
engineered_count = len(engineered_features)

# Only diagnosis codes are replaced by engineering
kept_diags = ['diag_1', 'diag_2', 'diag_3']
raw_features_dropped_after_engineering = len(kept_diags)  # Only 3 diagnosis columns replaced
raw_features_kept_as_is = keep_count - raw_features_dropped_after_engineering

final_feature_count = raw_features_kept_as_is + engineered_count

print(f"\nRaw features (from Section II.2):")
print(f"  • Total KEEP features: {keep_count}")
print(f"  • Raw features KEPT AS-IS: {raw_features_kept_as_is}")
print(f"    - Medication columns: {len(kept_meds)} (individual features preserved)")
print(f"    - Utilization columns: {len(util_features)} (raw counts preserved)")
print(f"    - Other features: {raw_features_kept_as_is - len(kept_meds) - len(util_features)}")
print(f"  • Raw features REPLACED by engineering: {raw_features_dropped_after_engineering}")
print(f"    - Diagnosis columns (diag_1, diag_2, diag_3): 3 → 27 binary flags")
print(f"\nEngineered features (new): {engineered_count}")
print(f"  • Diagnosis categories: 27")
print(f"  • Age stratification: 6")
print(f"  • Clinical complexity (optional): 3")
print(f"\n{'='*60}")
print(f"TOTAL FEATURES FOR MODELING: {final_feature_count}")
print(f"{'='*60}")

print("\n✓ Feature engineering specifications complete")
print("  → Diagnosis codes aggregated into 27 clinical category flags")
print("  → Medication features KEPT INDIVIDUAL (preserve drug-specific effects)")
print("  → Utilization features KEPT RAW (preserve encounter type patterns)")
print("  → Age stratified into 6 granular buckets (capture non-linear effects)")

FEATURE ENGINEERING SPECIFICATIONS

1. DIAGNOSIS AGGREGATION (9 clinical categories)

Source features: diag_1, diag_2, diag_3 (717-790 unique values each)

Engineered features:
  • diag_1_circulatory             = ICD-9 390-459, 785 (heart disease, hypertension, stroke)
  • diag_1_respiratory             = ICD-9 460-519, 786 (COPD, pneumonia, asthma)
  • diag_1_diabetes                = ICD-9 250.xx (diabetes complications)
  • diag_1_digestive               = ICD-9 520-579, 787 (GI disorders)
  • diag_1_injury                  = ICD-9 800-999 (fractures, trauma)
  • diag_1_musculoskeletal         = ICD-9 710-739 (arthritis, back pain)
  • diag_1_genitourinary           = ICD-9 580-629, 788 (kidney, urinary)
  • diag_1_neoplasms               = ICD-9 140-239 (cancer)
  • diag_1_other                   = All other ICD-9 codes

  [Repeat same categories for diag_2 and diag_3]
  Total diagnosis features: 9 × 3 = 27 binary flags

2. MEDICATION FEATURES (keep individual columns)

Source fea

## 4. Visualizations

First summary of all final features with preprocessing specifications.

In [32]:
# List all final features for modeling with preprocessing specs
print("="*100)
print("FINAL FEATURE LIST FOR MODELING (with preprocessing specifications)")
print("="*100)

# Get raw features kept as-is (exclude diagnosis codes diag_1, diag_2, diag_3)
raw_keep_features = feature_df[feature_df['Decision']=='KEEP']['Feature'].tolist()
diag_cols = ['diag_1', 'diag_2', 'diag_3']
raw_keep_final = [f for f in raw_keep_features if f not in diag_cols]

print(f"\n{len(raw_keep_final)} RAW FEATURES (kept as-is):")
print("-" * 100)
for i, feat in enumerate(sorted(raw_keep_final), 1):
    feat_info = feature_df[feature_df['Feature']==feat].iloc[0]
    print(f"  {i:2d}. {feat:30s} ({feat_info['Type']:12s})")
    print(f"      Predictive Strength: {feat_info['Predictive_Strength']:.4f}")
    print(f"      Preprocessing: {feat_info['Processing_Specs']}")
    print()

# Engineered features from diagnosis codes
print(f"\n27 DIAGNOSIS CATEGORY FEATURES (engineered from diag_1, diag_2, diag_3):")
print("-" * 100)
print("Type: Binary flags")
print("Preprocessing: No further encoding needed (already binary)")
print("\nFeatures:")
diag_categories = ['circulatory', 'respiratory', 'diabetes', 'digestive', 'injury', 
                   'musculoskeletal', 'genitourinary', 'neoplasms', 'other']
for diag_num in [1, 2, 3]:
    print(f"\n  From diag_{diag_num}:")
    for cat in diag_categories:
        print(f"    • diag_{diag_num}_{cat}")

# Age stratification features
print(f"\n\n6 AGE STRATIFICATION FEATURES (engineered from age):")
print("-" * 100)
age_features_info = [
    ('age_bucket', 'Ordinal', 'Ordinal encoding (0-9) | Preserves natural order'),
    ('age_young', 'Binary', 'Binary flag | No encoding needed'),
    ('age_adult', 'Binary', 'Binary flag | No encoding needed'),
    ('age_middle', 'Binary', 'Binary flag | No encoding needed'),
    ('age_senior', 'Binary', 'Binary flag | No encoding needed'),
    ('age_elderly', 'Binary', 'Binary flag | No encoding needed')
]
for i, (feat, feat_type, prep) in enumerate(age_features_info, 1):
    print(f"  {i}. {feat:20s} ({feat_type})")
    print(f"     Preprocessing: {prep}")

# Summary
total_features = len(raw_keep_final) + 27 + 6
print(f"\n\n{'='*100}")
print(f"TOTAL FEATURES: {total_features}")
print(f"  • Raw features (kept as-is): {len(raw_keep_final)}")
print(f"  • Diagnosis categories: 27")
print(f"  • Age stratification: 6")
print(f"{'='*100}\n")

FINAL FEATURE LIST FOR MODELING (with preprocessing specifications)

23 RAW FEATURES (kept as-is):
----------------------------------------------------------------------------------------------------
   1. A1Cresult                      (Categorical )
      Predictive Strength: 0.0047
      Preprocessing: Impute: mode | Create missing_indicator | Encode: one-hot

   2. admission_type_id              (Numerical   )
      Predictive Strength: 0.0117
      Preprocessing: Outliers: IQR winsorize + range validation

   3. age                            (Categorical )
      Predictive Strength: 0.0339
      Preprocessing: Protected: preserve original values | Encode: ordinal (preserve order)

   4. change                         (Categorical )
      Predictive Strength: 0.0195
      Preprocessing: Encode: one-hot

   5. diabetesMed                    (Categorical )
      Predictive Strength: 0.0271
      Preprocessing: Encode: one-hot

   6. discharge_disposition_id       (Numerical   )
    

Visualizations include: